In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, adjusted_mutual_info_score

In [2]:
# Load sample dataset
data = load_wine()
X, y = data.data, data.target  # Using the target for evaluation

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
# Define pipeline with preprocessing, scaling, PCA, and clustering model
pipeline = Pipeline([
    ('scaler', StandardScaler()),   # Feature Scaling
    ('pca', PCA()),                 # Principal Component Analysis
    ('clustering', KMeans())        # Clustering Model (KMeans)
])


In [4]:
# Define parameter grid for Grid Search
param_grid = {
    'pca__n_components': [2, 3, 5],                     # Number of PCA components
    'clustering__n_clusters': [2, 3, 4, 5, 6, 7],       # Number of clusters for KMeans
    'clustering__init': ['k-means++', 'random'],        # Initialization method for KMeans
    'clustering__max_iter': [300, 400, 500]             # Maximum number of iterations for KMeans
}

# Grid Search with Cross-Validation
# Note: Clustering does not inherently have a validation phase, so CV here is not the same as in supervised learning.
#       We evaluate by clustering stability using silhouette score on the training data.
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)  # y_train is used for evaluation, not for fitting

# Best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best Score (negative MSE):", grid_search.best_score_)

Best Parameters: {'clustering__init': 'k-means++', 'clustering__max_iter': 500, 'clustering__n_clusters': 3, 'pca__n_components': 3}
Best Score (negative MSE): -0.4005614657210402


/usr/local/lib/python3.10/dist-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [5]:
# Evaluate on test data
best_model = grid_search.best_estimator_
X_test_transformed = best_model.named_steps['scaler'].transform(X_test)
X_test_pca = best_model.named_steps['pca'].transform(X_test_transformed)
y_pred_test = best_model.named_steps['clustering'].predict(X_test_pca)

# Evaluation Metrics
silhouette_avg = silhouette_score(X_test_pca, y_pred_test)
ari = adjusted_rand_score(y_test, y_pred_test)
ami = adjusted_mutual_info_score(y_test, y_pred_test)

print(f"Silhouette Score: {silhouette_avg:.4f}")
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Adjusted Mutual Information: {ami:.4f}")


Silhouette Score: 0.4357
Adjusted Rand Index: 0.8251
Adjusted Mutual Information: 0.8449
